<div dir="rtl">

# الترم ٢ · الوحدة ٣: الشبكات العصبية

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-2/06-neural-networks.ipynb)

هنبني **نيورون واحد (Perceptron)** من الصفر بـ NumPy علشان نفهم إزاي بيتعلم، وبعدين نستخدم **شبكة كاملة** تتعرف على أرقام مكتوبة بخط الإيد.

</div>

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # small demo networks

<div dir="rtl">

## ١. النيورون

النيورون بياخد مدخلات، يضرب كل واحد في **وزن (weight)**، يجمعهم ويضيف **bias**، وبعدين **activation function** تقرر "يولّع ولا لأ".

$$ \text{output} = \text{activation}(w_1x_1 + w_2x_2 + b) $$

</div>

In [ ]:
def step(z):
    return (z >= 0).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(z, step(z), label="step")
plt.plot(z, sigmoid(z), label="sigmoid")
plt.plot(z, np.maximum(0, z) / 6, label="ReLU (scaled)")
plt.legend()
plt.title("Activation functions")
plt.show()

<div dir="rtl">

## ٢. نعلّم نيورون بوابة AND

| x1 | x2 | AND |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

**قاعدة التعلم:** لو الإجابة غلط، زحزح الأوزان شوية في اتجاه الصح: `w = w + lr * error * x`

</div>

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

def train_perceptron(X, y, epochs=20, lr=0.1, seed=0):
    rng = np.random.default_rng(seed)
    w, b = rng.normal(0, 0.5, X.shape[1]), 0.0
    history = []
    for epoch in range(epochs):
        errors = 0
        for xi, target in zip(X, y):
            pred = step(np.dot(w, xi) + b)
            error = target - pred
            w += lr * error * xi
            b += lr * error
            errors += int(error != 0)
        history.append(errors)
    return w, b, history

w, b, history = train_perceptron(X, y_and)
print("Weights:", w.round(2), "Bias:", round(b, 2))
print("Predictions:", step(X @ w + b), "Target:", y_and)

plt.figure(figsize=(6, 3))
plt.plot(history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mistakes")
plt.title("The neuron learns: mistakes go to zero")
plt.show()

<div dir="rtl">

## ٣. مشكلة XOR: نيورون واحد مش كفاية

XOR = "واحد بس منهم صح". جرّب:

</div>

In [ ]:
y_xor = np.array([0, 1, 1, 0])
w, b, history = train_perceptron(X, y_xor, epochs=50)
print("XOR predictions:", step(X @ w + b), "Target:", y_xor)
print("Mistakes in last epoch:", history[-1])

<div dir="rtl">

النيورون الواحد بيرسم **خط مستقيم** واحد، ومفيش خط واحد يفصل نقط XOR. الحل: **طبقات** من النيورونات. ده اللي اسمه **Deep Learning**.

</div>

In [ ]:
xor_net = MLPClassifier(hidden_layer_sizes=(4,), activation="tanh", solver="lbfgs", random_state=1, max_iter=2000)
xor_net.fit(X, y_xor)
print("Network with a hidden layer:", xor_net.predict(X))

<div dir="rtl">

## ٤. Backpropagation في جملتين

1. الشبكة بتتوقع، ونحسب **الخطأ**.
2. نرجع بالخطأ **لورا** طبقة طبقة، ونعدّل كل وزن على قد مساهمته في الغلط.

بنكرر ده آلاف المرات. ده بالظبط اللي `fit` بتعمله تحت.

</div>

<div dir="rtl">

## ٥. شبكة حقيقية: التعرف على الأرقام المكتوبة بخط الإيد

١٧٩٧ صورة لأرقام من ٠ لـ ٩، كل صورة ٨×٨ بكسل.

</div>

In [ ]:
digits = load_digits()
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, img, label in zip(axes.flat, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.show()
print("One image as numbers:\n", digits.images[0].astype(int))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(digits.data / 16.0, digits.target,
                                                    test_size=0.25, random_state=0, stratify=digits.target)
net = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=0)
net.fit(X_train, y_train)
print(f"Test accuracy: {net.score(X_test, y_test):.1%}")

plt.figure(figsize=(6, 3))
plt.plot(net.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Loss (error)")
plt.title("Training: the error goes down")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_estimator(net, X_test, y_test, cmap="Blues")
plt.show()

<div dir="rtl">

### الشبكة بتغلط فين؟

</div>

In [ ]:
pred = net.predict(X_test)
wrong = np.where(pred != y_test)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(1.4 * len(wrong), 2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"{y_test[i]}->{pred[i]}", color="red")
    ax.axis("off")
plt.show()

<div dir="rtl">

## 🏋️ تمرين ١: بوابة OR
درّب الـ perceptron على OR (الناتج 1 لو أي واحد فيهم 1).

<details><summary>الحل</summary>

```python
y_or = np.array([0, 1, 1, 1])
w, b, _ = train_perceptron(X, y_or)
print(step(X @ w + b), "target:", y_or)
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢: حجم الشبكة
جرّب `hidden_layer_sizes` = `(4,)`, `(16,)`, `(64, 32)`, `(128, 64, 32)`. الدقة بتتغير إزاي؟

<details><summary>الحل</summary>

```python
for size in [(4,), (16,), (64, 32), (128, 64, 32)]:
    m = MLPClassifier(hidden_layer_sizes=size, max_iter=300, random_state=0).fit(X_train, y_train)
    print(size, f"{m.score(X_test, y_test):.1%}")
# الشبكة الصغيرة جداً ما بتقدرش تتعلم كويس، وبعد حجم معين الزيادة ما بتفرقش كتير.
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🔭 الخطوة الجاية: TensorFlow / Keras

الشبكات الكبيرة (صور ملونة، كلام) بنبنيها بمكتبات زي **Keras**. نفس الفكرة بالظبط:

```python
from tensorflow import keras
model = keras.Sequential([
    keras.layers.Dense(64, activation="relu", input_shape=(64,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=20, validation_split=0.1)
```

جرّبه في Colab (TensorFlow متسطب فيه جاهز). وجرّب كمان [TensorFlow Playground](https://playground.tensorflow.org) علشان تشوف الشبكة بتتعلم قدام عينك.

</div>